# Final Modeling Experiments

In [80]:
import numpy as np
import pandas as pd
import optuna
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve
from preprocessing import import_data, group_diagnoses, group_medications, final_engineered, features
from scoring import score

In [81]:
STUDY_NAME = "logistic_f2_simple_v2"
STORAGE = "sqlite:///logistic_regression_Version2_simple.db"

Xt,yt,Xv,yv = import_data()

Importing train/validation data...
Finished the import.


In [82]:
Xt,Xv,all_features=final_engineered(Xt,Xv)

In [83]:
study = optuna.create_study(study_name=STUDY_NAME, storage=STORAGE,
                            direction="maximize", load_if_exists=True,
                            sampler=optuna.samplers.TPESampler(n_startup_trials=30))

[I 2026-10-30 20:21:22,257] Using an existing study with `study_name='logistic_f2_simple_v2'` instead of creating a new one.


In [84]:
# This is a copy and paste of the objective function from the other notebook.

def objective(trial, return_model=False):
    mode = trial.suggest_categorical("feature_set", ["baseline", "all", "selected"])
    chosen = baseline_features if mode == "baseline" else all_features
    if mode == "selected":
        chosen = [c for c in all_features if trial.suggest_categorical("use_" + c, [True, False])]
    if not chosen:
        return 0.0
    nums = Xt[chosen].select_dtypes(include="number").columns.tolist()
    cats = [c for c in chosen if c not in nums]
    scaler = trial.suggest_categorical("scaler", ["standard", "robust"])
    num = Pipeline([
        ("impute", SimpleImputer(strategy=trial.suggest_categorical("num_impute", ["median", "mean", "constant"]),
                                fill_value=0, keep_empty_features=True,
                                add_indicator=trial.suggest_categorical("missing_indicators", [False, True]))),
        ("scale", StandardScaler() if scaler == "standard" else RobustScaler()),
    ])
    cat = Pipeline([
        ("impute", SimpleImputer(strategy=trial.suggest_categorical("cat_impute", ["constant", "most_frequent"]),
                                fill_value="Missing", keep_empty_features=True)),
        ("encode", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                 min_frequency=trial.suggest_categorical("min_frequency", [1, 10, 50, 100, 250]))),
    ])
    weight_mode = trial.suggest_categorical("class_weight", ["none", "balanced", "custom"])
    weight = {0: 1, 1: trial.suggest_float("positive_weight", 0.5, 40, log=True)} if weight_mode == "custom" else (
        "balanced" if weight_mode == "balanced" else None)
    ratio = trial.suggest_categorical("l1_ratio", [0.0, 0.25, 0.5, 0.75, 1.0])
    model = Pipeline([
        ("prep", ColumnTransformer([("num", num, nums), ("cat", cat, cats)])),
        ("model", LogisticRegression(C=trial.suggest_float("C", 1e-4, 100, log=True),
            l1_ratio=ratio, solver="lbfgs" if ratio == 0 else "saga",
            class_weight=weight, max_iter=5000, random_state=42)),
    ])
    model.fit(Xt[chosen], yt)
    probabilities = model.predict_proba(Xv[chosen])[:, 1]
    precision, recall, thresholds = precision_recall_curve(yv, probabilities)
    f2 = 5 * precision[:-1] * recall[:-1] / (4 * precision[:-1] + recall[:-1] + 1e-15)
    best = int(f2.argmax())
    trial.set_user_attr("features", chosen)
    trial.set_user_attr("n_features", len(chosen))
    trial.set_user_attr("threshold", float(thresholds[best]))
    if return_model:
        return model, chosen, float(thresholds[best])
    return float(f2[best])

In [85]:
best_model, chosen_features, threshold = objective(optuna.trial.FixedTrial(study.best_params), return_model=True)

In [86]:
best_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](131,)","['time_in_hospital','num_lab_procedures','num_procedures',..., 'age_with_discharge_disposition_id','insulin_with_change', 'admission_type_id_with_admission_source_id']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,131
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedco

In [87]:
chosen_features

['time_in_hospital',
 'num_lab_procedures',
 'num_procedures',
 'num_medications',
 'number_outpatient',
 'number_emergency',
 'number_inpatient',
 'number_diagnoses',
 'race',
 'gender',
 'age',
 'weight',
 'admission_type_id',
 'discharge_disposition_id',
 'admission_source_id',
 'payer_code',
 'medical_specialty',
 'diag_1',
 'diag_2',
 'diag_3',
 'max_glu_serum',
 'a1cresult',
 'metformin',
 'repaglinide',
 'nateglinide',
 'chlorpropamide',
 'glimepiride',
 'acetohexamide',
 'glipizide',
 'glyburide',
 'tolbutamide',
 'pioglitazone',
 'rosiglitazone',
 'acarbose',
 'miglitol',
 'troglitazone',
 'tolazamide',
 'examide',
 'citoglipton',
 'insulin',
 'glyburide-metformin',
 'glipizide-metformin',
 'glimepiride-pioglitazone',
 'metformin-rosiglitazone',
 'metformin-pioglitazone',
 'change',
 'diabetesmed',
 'diag_1_group',
 'diag_2_group',
 'diag_3_group',
 'alpha_glucosidase_inhibitor',
 'biguanide',
 'biguanide_sulfonylurea_gen2',
 'biguanide_thiazolidinedione',
 'meglitinide',
 'su

In [88]:
# The following code reports the weights and the feature names.

prep, model = best_model.named_steps["prep"], best_model.named_steps["model"]

def names_from(t, c):
    c = list(Xt.columns[c]) if isinstance(c, slice) else [c] if isinstance(c, str) else list(c)
    for s in reversed([s for _, s in t.steps] if hasattr(t, "steps") else [t]):
        for m in ("get_feature_names_out", "get_feature_names"):
            for a in ((c,), ()):
                try: return list(getattr(s, m)(*a))
                except Exception: pass
    return c  # passthrough / 1:1 transformer

names = [n for k, t, c in prep.transformers_ if not (isinstance(t, str) and t == "drop")
         for n in names_from(t, [Xt.columns[i] for i in c] if k == "remainder" and isinstance(t, str) else c)]

print(f"Model Intercept (Bias): {model.intercept_[0]:.4f}\n")

weights_df = pd.DataFrame({"Feature Name": names, "Weight (Coefficient)": model.coef_[0]}).sort_values("Weight (Coefficient)", ascending=False, ignore_index=True)
display(weights_df)

Model Intercept (Bias): 0.0002



,Feature Name,Weight (Coefficient)
0,discharge_disposition_id_Discharged/transferre...,0.949986
1,discharge_disposition_id_Discharged/transferre...,0.599240
2,diag_1_V58,0.592114
3,discharge_disposition_id_Discharged/transferre...,0.453700
4,discharge_disposition_id_Discharged/transferre...,0.352571
...,...,...
2807,discharge_disposition_id_Hospice / medical fac...,-0.250697
2808,num_procedures_x_num_medications,-0.259324
2809,age_with_discharge_disposition_id_[70-80)|Expired,-0.268963
2810,discharge_disposition_id_Hospice / home,-0.303709


In [89]:
display(weights_df.sort_values("Weight (Coefficient)", key=abs, ascending=False, ignore_index=True).head(20))

,Feature Name,Weight (Coefficient)
0,discharge_disposition_id_Expired,-1.604926
1,discharge_disposition_id_Discharged/transferre...,0.949986
2,discharge_disposition_id_Discharged/transferre...,0.599240
3,diag_1_V58,0.592114
4,discharge_disposition_id_Discharged/transferre...,0.453700
5,discharge_disposition_id_Discharged/transferre...,0.352571
6,number_emergency_log1p,0.319139
7,discharge_disposition_id_Hospice / home,-0.303709
8,age_with_discharge_disposition_id_[70-80)|Expired,-0.268963
9,num_procedures_x_num_medications,-0.259324


In [111]:
# drop zero weights

chosen_weights = weights_df[~ weights_df["Weight (Coefficient)"].eq(0)]

chosen_weights = chosen_weights[~ (abs(chosen_weights["Weight (Coefficient)"]) <= 0.1)]

chosen_weights

,Feature Name,Weight (Coefficient)
0,discharge_disposition_id_Discharged/transferre...,0.949986
1,discharge_disposition_id_Discharged/transferre...,0.599240
2,diag_1_V58,0.592114
3,discharge_disposition_id_Discharged/transferre...,0.453700
4,discharge_disposition_id_Discharged/transferre...,0.352571
...,...,...
2807,discharge_disposition_id_Hospice / medical fac...,-0.250697
2808,num_procedures_x_num_medications,-0.259324
2809,age_with_discharge_disposition_id_[70-80)|Expired,-0.268963
2810,discharge_disposition_id_Hospice / home,-0.303709


In [162]:
display(chosen_weights.head(40))
chosen_weights.tail(40)

,Feature Name,Weight (Coefficient)
0,discharge_disposition_id_Discharged/transferre...,0.949986
1,discharge_disposition_id_Discharged/transferre...,0.599240
2,diag_1_V58,0.592114
3,discharge_disposition_id_Discharged/transferre...,0.453700
4,discharge_disposition_id_Discharged/transferre...,0.352571
5,number_emergency_log1p,0.319139
6,diag_3_250.6,0.230778
7,number_inpatient_log1p,0.211307
8,diag_1_250.6,0.208280
9,diag_3_403,0.206414


,Feature Name,Weight (Coefficient)
2772,weight_Missing,-0.101037
2773,admission_type_id_Not Available,-0.102885
2774,admission_source_id_Transfer from a Skilled Nu...,-0.107804
2775,discharge_disposition_id_Not Mapped,-0.110624
2776,diag_1_518,-0.111851
2777,diag_3_414,-0.112806
2778,num_lab_procedures_x_number_emergency,-0.113610
2779,diag_3_272,-0.114275
2780,diag_1_group_with_insulin_Musculoskeletal|No,-0.115406
2781,repaglinide_No,-0.116718


In [98]:
probs = best_model.predict_proba(Xv)[:,1]
score(yv, y_pred=None, proba=probs, threshold=0.65989)

Accuracy: 0.846
Confusion matrix:
 [[13359  1107]
 [ 1393   424]]
              precision    recall  f1-score   support

           0       0.91      0.92      0.91     14466
           1       0.28      0.23      0.25      1817

    accuracy                           0.85     16283
   macro avg       0.59      0.58      0.58     16283
weighted avg       0.84      0.85      0.84     16283

F1 @0.65989: 0.253
F2 @0.65989: 0.241
ROC-AUC: 0.6794
PR-AUC:  0.2217

Best-threshold F1: 0.28734 at threshold 0.54561.

Best-threshold F2: 0.43078 at threshold 0.43851.


# Fit Models

In [ ]:
features_to_keep = ["discharge_disposition_id", 
                    "diag_1", 
                    "diag_3", 
                    "number_emergency_log1p", 
                    "number_inpatient", 
                    "age_with_discharge_disposition_id", 
                    # "medical_specialty", 
                    # "weight", 
                    # "glyburide", 
                    "admission_source_id",
                    "num_procedures_x_num_medications", 
                    "payer_code", 
                    "num_lab_procedures_x_num_procedures", 
                    "number_emergency_bin", 
                    "admission_type_id_with_admission_source_id", 
                    "num_lab_procedures_x_number_emergency", 
                    "number_inpatient_x_number_diagnoses",
                    # "number_inpatient_share", 
                    # "diag_2_group", 
                    # "repaglinide", 
                    "diag_1_group_with_insulin", 
                    "diag_1_group_with_a1cresult",
                    "number_emergency",
                    "number_diagnoses",
                    "num_medications",
                    ]

Xtt = Xt[features_to_keep] 
Xvt = Xv[features_to_keep]

num_cols = Xtt.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = Xtt.select_dtypes(include=['object','category']).columns.tolist()


In [193]:
num = Pipeline([("imp", SimpleImputer(strategy="mean",fill_value=0,keep_empty_features=True)),("scale", StandardScaler())])

cat = Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Missing", keep_empty_features=True)), 
                ("oh", OneHotEncoder(handle_unknown="infrequent_if_exist",min_frequency=1))])

prep = ColumnTransformer([("num", num, num_cols),
                          ("cat", cat, cat_cols)])

model = Pipeline([
        ("prep", ColumnTransformer([("num", num, num_cols), ("cat", cat, cat_cols)])),
        ("model", LogisticRegression(C=0.015731183591896788,
            l1_ratio=0.25, solver="saga",
            class_weight='balanced', max_iter=2500, random_state=42)),
    ])

model.fit(Xtt,yt)
probs = model.predict_proba(Xvt)[:,1]
score(yv, y_pred=None, proba=probs, threshold=0.65989)


Accuracy: 0.849
Confusion matrix:
 [[13420  1046]
 [ 1413   404]]
              precision    recall  f1-score   support

           0       0.90      0.93      0.92     14466
           1       0.28      0.22      0.25      1817

    accuracy                           0.85     16283
   macro avg       0.59      0.58      0.58     16283
weighted avg       0.83      0.85      0.84     16283

F1 @0.65989: 0.247
F2 @0.65989: 0.232
ROC-AUC: 0.6753
PR-AUC:  0.2172

Best-threshold F1: 0.28471 at threshold 0.57446.

Best-threshold F2: 0.42618 at threshold 0.43689.
